In [ ]:
"""
Extrae la serie de evapotranspiración real (ET) quincenal para un punto,
usando doble fuente en Earth Engine:

1. MODIS/061/MOD16A2GF (Gap-Filled) - fuente primaria, con relleno de
   huecos, pero "year-end gap-filled": el año en curso puede no tener
   NINGÚN dato hasta que cierre.
2. MODIS/061/MOD16A2 (no gap-filled) - fuente de respaldo, casi
   tiempo real, para las quincenas donde la primaria todavía no
   publicó. Sin relleno de huecos, así que puede seguir faltando algún
   composite individual por nubosidad persistente.

CÓMO LEER EL RESULTADO
-----------------------
et_total_mm    Agua total perdida por evaporación + transpiración en
               esa quincena (según la vegetación actual del pixel, no
               necesariamente el cultivo que planeás poner - ver nota
               abajo).
fuente         De qué producto salió el dato: 'GF', 'no-GF', o
               'sin_dato' si ninguna de las dos fuentes tenía registro
               para esa quincena.

USO EN EL BALANCE HÍDRICO: junto con precipitation_profile.py (entrada)
y soil_hydraulics.py (capacidad de almacenamiento), esto cierra el
balance:
    Δalmacenamiento = precipitación - ET - excedente (cuando supera AWC)

LIMITACIÓN IMPORTANTE: ET real refleja la vegetación EXISTENTE en el
pixel hoy, no la del cultivo que planees plantar (que puede consumir
más o menos agua). Sirve como piso orientativo para decidir si hace
falta capacidad de riego instalada, no como dimensionamiento exacto
del sistema.

LIMITACIÓN TÉCNICA (composites): ambos productos vienen en composites
de 8 días, que no calzan exactamente con los bordes de las quincenas
(1-15, 16-fin de mes) -> puede haber un pequeño corrimiento en los
bordes. Para el objetivo de detectar déficit general, esto no afecta
la conclusión.

CITAS (para metodología de tesis):
Running, S., Mu, Q., & Zhao, M. (2021). MODIS/Terra Net
Evapotranspiration Gap-Filled 8-Day L4 Global 500m SIN Grid V061
[Data set]. NASA EOSDIS LP DAAC. https://doi.org/10.5067/MODIS/MOD16A2GF.061
Running, S., Mu, Q., & Zhao, M. (2021). MODIS/Terra Net
Evapotranspiration 8-Day L4 Global 500m SIN Grid V061 [Data set].
NASA EOSDIS LP DAAC. https://doi.org/10.5067/MODIS/MOD16A2.061
"""
from datetime import date, datetime
from pathlib import Path
import pandas as pd
import ee

from test_period_utils import build_biweekly_periods


def _get_mod16_band_biweekly(lat, lon, band, column_name, start_date="2016-01-01", end_date=None):
    """
    Función genérica compartida: extrae cualquier banda de MOD16A2/GF
    (ET o PET) con la misma estrategia de doble fuente. Usada tanto por
    get_et_biweekly() (band='ET') como por spei_profile.py (band='PET').
    """
    start = datetime.strptime(start_date, "%Y-%m-%d").date()
    end = datetime.strptime(end_date, "%Y-%m-%d").date() if end_date else date.today()

    periods = build_biweekly_periods(start, end)
    print(f"[DEBUG] {len(periods)} quincenas a procesar ({band}), desde {start} hasta {end}")

    point = ee.Geometry.Point([lon, lat])

    col_gf = (
        ee.ImageCollection('MODIS/061/MOD16A2GF')
        .select(band)
        .map(lambda img: img.multiply(0.1).copyProperties(img, ['system:time_start']))
    )
    col_fallback = (
        ee.ImageCollection('MODIS/061/MOD16A2')
        .select(band)
        .map(lambda img: img.multiply(0.1).copyProperties(img, ['system:time_start']))
    )

    ee_periods = ee.List([
        {'label': label, 'start': str(p_start), 'end': str(p_end)}
        for label, p_start, p_end in periods
    ])

    def compute_period(period):
        period = ee.Dictionary(period)
        p_start = ee.Date(period.get('start'))
        p_end = ee.Date(period.get('end'))

        filtered_gf = col_gf.filterDate(p_start, p_end)
        filtered_fb = col_fallback.filterDate(p_start, p_end)

        has_gf = filtered_gf.size().gt(0)
        has_fb = filtered_fb.size().gt(0)

        img_gf = ee.Image(ee.Algorithms.If(
            has_gf,
            filtered_gf.sum().rename(column_name).set('fuente', 'GF'),
            ee.Image.constant(0).rename(column_name).selfMask().set('fuente', 'sin_dato')
        ))
        img_fb = ee.Image(ee.Algorithms.If(
            has_fb,
            filtered_fb.sum().rename(column_name).set('fuente', 'no-GF'),
            ee.Image.constant(0).rename(column_name).selfMask().set('fuente', 'sin_dato')
        ))

        final_img = ee.Image(ee.Algorithms.If(has_gf, img_gf, img_fb))

        stats = final_img.reduceRegion(
            reducer=ee.Reducer.first(),
            geometry=point,
            scale=500,
            maxPixels=1e9
        )

        return ee.Feature(
            None,
            stats
            .set('label', period.get('label'))
            .set('periodo_inicio', p_start.format('YYYY-MM-dd'))
            .set('periodo_fin', p_end.advance(-1, 'day').format('YYYY-MM-dd'))
            .set('fuente', final_img.get('fuente'))
        )

    features = ee.FeatureCollection(ee_periods.map(compute_period))
    result = features.getInfo()

    rows = []
    for f in result['features']:
        props = f['properties']
        rows.append({
            'periodo_inicio': props.get('periodo_inicio'),
            'periodo_fin': props.get('periodo_fin'),
            'label': props.get('label'),
            'lat': lat,
            'lon': lon,
            column_name: round(props.get(column_name), 2) if props.get(column_name) is not None else None,
            'fuente': props.get('fuente'),
        })

    df = pd.DataFrame(rows)
    df['periodo_inicio'] = pd.to_datetime(df['periodo_inicio'])
    df = df.sort_values('periodo_inicio').reset_index(drop=True)

    filas_nulas = df[column_name].isna().sum()
    if filas_nulas > 0:
        primeras_nulas = df[df[column_name].isna()]['label'].tolist()
        print(f"[AVISO] {filas_nulas} quincena(s) sin datos en ninguna fuente "
              f"(GF ni no-GF) para {band}: {primeras_nulas}")

    return df


def get_et_biweekly(lat, lon, start_date="2016-01-01", end_date=None):
    """Serie quincenal de evapotranspiración REAL (ET). Ver docstring del módulo."""
    return _get_mod16_band_biweekly(lat, lon, 'ET', 'et_total_mm', start_date, end_date)


def save_et_profile(df, out_prefix="et_biweekly", output_dir="../databases"):
    """
    Guarda la serie de evapotranspiración con timestamp en el nombre:
    {out_prefix}-vYYMMDDHHMMSS.csv (mismo patrón que el resto del pipeline)
    """
    timestamp = datetime.now().strftime("%y%m%d%H%M%S")
    filename = f"{out_prefix}-v{timestamp}.csv"

    output_path = Path(output_dir)
    output_path.mkdir(parents=True, exist_ok=True)
    out_path = output_path / filename

    df.to_csv(out_path, index=False)
    print(f"CSV guardado en {out_path} ({df.shape[0]}x{df.shape[1]})")
    return out_path


if __name__ == "__main__":
    # Sugarcane_QLD
    LAT = -19.689669877950884
    LON = 147.22717515914223

    df = get_et_biweekly(LAT, LON, start_date="2016-01-01")
    out_path = save_et_profile(df)
    print(df.head(10))

In [ ]:
"""
Calcula el SPEI (Standardized Precipitation Evapotranspiration Index)
en múltiples escalas temporales (1, 3, 6, 12 "meses", equivalentes a
2, 6, 12, 24 quincenas) para un punto, siguiendo Vicente-Serrano,
Beguería & López-Moreno (2010).

PET: método de Thornthwaite (el mismo que usa el paper original de
Vicente-Serrano et al. 2010 y el paquete oficial 'SPEI' en R), corregido
para reflejar la fórmula real:
- El índice de calor anual (I) se calcula a partir de las temperaturas
  NORMALES climatológicas de los 12 meses del año (promedio histórico
  de toda la serie por mes-calendario), no de la temperatura de un
  solo período multiplicada por 12.
- Se corrige por duración real de horas de luz según latitud y época
  del año (geometría solar, misma fórmula que FAO-56 Allen et al.,
  1998, ec. 25/34 para horas de luz), en vez de asumir un factor fijo.

AGRUPACIÓN PARA EL AJUSTE DE LA DISTRIBUCIÓN: cada "quincena-calendario"
(ej. todas las "03_Q1" de los 10 años) se ajusta por separado, de forma
EXPLÍCITA (no inferida por la librería) -> se sabe con certeza qué se
agrupó con qué.

CÓMO LEER EL RESULTADO
-----------------------
balance_mm         Precipitación - PET de esa quincena (mm, sin
                    estandarizar).
spei_1m/3m/6m/12m  Índice estandarizado en cada escala temporal:
                        >= 2.0          extremadamente húmedo
                        1.5 a 1.99      muy húmedo
                        1.0 a 1.49      moderadamente húmedo
                       -0.99 a 0.99     normal
                       -1.49 a -1.0     sequía moderada
                       -1.99 a -1.5     sequía severa
                       <= -2.0          sequía extrema
                    spei_1m = déficit de corto plazo (decisión
                    operativa de riego inmediato); spei_12m = sequía
                    estructural (decisión de viabilidad del lugar).

LIMITACIÓN A DOCUMENTAR: el SPEI "de libro" se calcula a escala
mensual; acá se adapta a resolución quincenal (24 "meses-calendario"
por año en vez de 12). Con 10 años de historia, cada grupo de
quincena-calendario tiene ~10 muestras para el ajuste de la
distribución - muestra chica para climatología (30+ años es lo ideal).

REQUIERE: pip install standard-precip

CITA: Vicente-Serrano, S.M., Beguería, S., & López-Moreno, J.I. (2010).
A multi-scalar drought index sensitive to global warming: The
Standardized Precipitation Evapotranspiration Index. Journal of
Climate, 23(7), 1696-1718.
"""
import math
from datetime import datetime
from pathlib import Path
import pandas as pd
from standard_precip import spi

from precipitation_profile import get_precipitation_biweekly
from temperature_profile import get_temperature_biweekly


def _day_length_hours(day_of_year, lat_deg):
    """
    Horas de luz solar para un día del año y latitud dados, vía
    geometría solar (misma fórmula que FAO-56, Allen et al. 1998).
    """
    lat_rad = math.radians(lat_deg)
    declinacion = 0.409 * math.sin(2 * math.pi / 365 * day_of_year - 1.39)
    x = -math.tan(lat_rad) * math.tan(declinacion)
    x = max(-1, min(1, x))  # clip por latitudes extremas (día polar/noche polar)
    angulo_horario_ocaso = math.acos(x)
    return (24 / math.pi) * angulo_horario_ocaso


def _thornthwaite_pet(df_temp, lat):
    """
    Calcula PET quincenal por el método de Thornthwaite corregido.
    df_temp: DataFrame con columnas periodo_inicio, label, media_C
              (salida de get_temperature_biweekly)
    """
    df = df_temp.copy()
    df['periodo_fin'] = pd.to_datetime(df['periodo_fin'])
    df['mes'] = df['periodo_inicio'].dt.month
    df['dia_del_anio'] = df['periodo_inicio'].dt.dayofyear
    df['dias_periodo'] = (df['periodo_fin'] - df['periodo_inicio']).dt.days + 1

    # Índice de calor anual (I): usando temperaturas NORMALES por
    # mes-calendario (promedio histórico de toda la serie), no la
    # temperatura de un solo período repetida
    normales_mensuales = df.groupby('mes')['media_C'].mean()
    indices_calor_mensuales = normales_mensuales.apply(
        lambda t: (t / 5) ** 1.514 if t > 0 else 0
    )
    I = indices_calor_mensuales.sum()
    a = (6.75e-7 * I**3) - (7.71e-5 * I**2) + (1.792e-2 * I) + 0.49239

    def pet_periodo(row):
        t = row['media_C']
        if t is None or pd.isna(t) or t <= 0 or I == 0:
            return 0.0
        pet_base = 16 * (10 * t / I) ** a  # PET de referencia (mes de 30 días, 12h luz)
        n_horas_luz = _day_length_hours(row['dia_del_anio'], lat)
        correccion = (n_horas_luz / 12) * (row['dias_periodo'] / 30)
        return pet_base * correccion

    df['pet_total_mm'] = df.apply(pet_periodo, axis=1).round(2)
    return df[['periodo_inicio', 'periodo_fin', 'label', 'pet_total_mm']]


def _quincena_calendario_id(label):
    """'2019-03_Q1' -> 5 (id 1-24, independiente del año, para agrupar
    la misma época de años distintos en el ajuste de la distribución)."""
    mes = int(label.split('-')[1].split('_')[0])
    quincena = 1 if label.endswith('Q1') else 2
    return (mes - 1) * 2 + quincena


def get_spei_biweekly(lat, lon, start_date="2016-01-01", end_date=None):
    """
    lat, lon: coordenadas del punto
    start_date, end_date: rango de fechas (str "YYYY-MM-DD"); end_date=None -> hoy

    Calcula SPEI en 4 escalas: 1, 3, 6 y 12 "meses" (2, 6, 12 y 24
    quincenas respectivamente).
    """
    df_precip = get_precipitation_biweekly(lat, lon, start_date, end_date)
    df_temp = get_temperature_biweekly(lat, lon, start_date, end_date)
    df_pet = _thornthwaite_pet(df_temp, lat)

    df = df_precip[['periodo_inicio', 'periodo_fin', 'label', 'lat', 'lon', 'precip_total_mm']].merge(
        df_pet[['label', 'pet_total_mm']], on='label', how='inner'
    )
    df['balance_mm'] = df['precip_total_mm'] - df['pet_total_mm']
    df['quincena_calendario'] = df['label'].apply(_quincena_calendario_id)

    filas_incompletas = df['balance_mm'].isna().sum()
    if filas_incompletas > 0:
        print(f"[AVISO] {filas_incompletas} quincena(s) sin precipitación disponible "
              f"-> balance_mm y spei quedan en NaN para esas filas.")

    df_completo = df.dropna(subset=['balance_mm']).copy()
    calculador = spi.SPI()

    # escala en "meses" -> número de quincenas equivalente (1 mes ~ 2 quincenas)
    escalas = {'spei_1m': 2, 'spei_3m': 6, 'spei_6m': 12, 'spei_12m': 24}

    for nombre_col, n_quincenas in escalas.items():
        try:
            resultado = calculador.calculate(
                df_completo,
                'periodo_inicio',
                'balance_mm',
                freq_col='quincena_calendario',
                scale=n_quincenas,
                fit_type='lmom',
                dist_type='fisk',  # log-logística
            )
            col_generada = [c for c in resultado.columns if c.startswith('balance_mm_scale')][0]
            df_completo[nombre_col] = resultado[col_generada]
        except Exception as e:
            print(f"[AVISO] No se pudo ajustar {nombre_col}: {e}. Se deja como NaN.")
            df_completo[nombre_col] = None

    df = df.merge(
        df_completo[['periodo_inicio'] + list(escalas.keys())],
        on='periodo_inicio', how='left'
    )

    return df[['periodo_inicio', 'periodo_fin', 'label', 'lat', 'lon',
               'precip_total_mm', 'pet_total_mm', 'balance_mm',
               'spei_1m', 'spei_3m', 'spei_6m', 'spei_12m']]


def save_spei_profile(df, out_prefix="spei_biweekly", output_dir="../databases"):
    """
    Guarda la serie de SPEI con timestamp en el nombre:
    {out_prefix}-vYYMMDDHHMMSS.csv (mismo patrón que el resto del pipeline)
    """
    timestamp = datetime.now().strftime("%y%m%d%H%M%S")
    filename = f"{out_prefix}-v{timestamp}.csv"

    output_path = Path(output_dir)
    output_path.mkdir(parents=True, exist_ok=True)
    out_path = output_path / filename

    df.to_csv(out_path, index=False)
    print(f"CSV guardado en {out_path} ({df.shape[0]}x{df.shape[1]})")
    return out_path


if __name__ == "__main__":
    # Finca Matanza
    LAT = 7.300921
    LON = -73.009794

    df = get_spei_biweekly(LAT, LON, start_date="2016-01-01")
    out_path = save_spei_profile(df)
    print(df.head(10))